In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification

# Mengenerate dataset dummy (1000 baris, 10 fitur) untuk simulasi UTS
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

In [2]:
# =======================================================
# ❌ KESALAHAN KONSEPTUAL: DATA LEAKAGE PADA PIPELINE
# =======================================================

scaler_wrong = StandardScaler()

# KESALAHAN FATAL: Fit_transform dilakukan pada KESELURUHAN data (X)
# Model "belajar" dari distribusi data test sebelum pemisahan
X_scaled_wrong = scaler_wrong.fit_transform(X) 

# Split data setelah di-scale
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_scaled_wrong, y, test_size=0.3, random_state=42
)

model_wrong = LogisticRegression()
model_wrong.fit(X_train_w, y_train_w)
y_pred_w = model_wrong.predict(X_test_w)

print("Akurasi dengan Pipeline SALAH (Bias/Optimis Palsu):", accuracy_score(y_test_w, y_pred_w))

Akurasi dengan Pipeline SALAH (Bias/Optimis Palsu): 0.8466666666666667


In [3]:
# =======================================================
# ✅ PERBAIKAN PIPELINE: MENCEGAH DATA LEAKAGE
# =======================================================

# 1. SPLIT DATA TERLEBIH DAHULU (Training dan Testing harus terisolasi)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler_right = StandardScaler()

# 2. Proses FIT HANYA dikenakan pada data Training
# (Mencari mean & std deviasi murni dari lingkungan training)
X_train_scaled = scaler_right.fit_transform(X_train_r)

# 3. Proses TRANSFORM dikenakan pada data Testing
# (Menerapkan hitungan dari training ke test, tanpa melakukan fit ulang)
X_test_scaled = scaler_right.transform(X_test_r)

model_right = LogisticRegression()
model_right.fit(X_train_scaled, y_train_r)
y_pred_r = model_right.predict(X_test_scaled)

print("Akurasi dengan Pipeline BENAR (Objektif/Valid):", accuracy_score(y_test_r, y_pred_r))

Akurasi dengan Pipeline BENAR (Objektif/Valid): 0.8466666666666667
